Surge-Sense: An ER Operations System built on RWFD of hospital patients' visits, the various metadata of those visits. The goal is to build a regressor to predict wait time. A sub-goal of this is building a forecaster to predict patient volume for a future day/time block; will be used as a feature in the regressor for predicting a patient's wait time. 

### Clean & Split
Taking our dataset and creating two data products for the respective models.

In [1]:
# hopefully fixing this reading csv problem; had the wrong environment to read in file with colab-—
from pathlib import Path
print(Path.cwd())
print(Path('er-data.csv').exists())

/content
False


In [2]:
from pathlib import Path
print(Path.cwd())
print(Path('er-data.csv').exists())

/content
False


In [3]:
# dependecies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# google colab upload fix
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
data = pd.read_csv('/content/drive/MyDrive/er-data.csv')
data.head()

,Patient Id,Patient Admission Date,Patient First Inital,Patient Last Name,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime
0,145-39-5406,20-03-2024 08:47,H,Glasspool,M,69,White,NaN,False,10.0,39
1,316-34-3057,15-06-2024 11:29,X,Methuen,M,4,Native American/Alaska Native,NaN,True,NaN,27
2,897-46-3852,20-06-2024 09:13,P,Schubuser,F,56,African American,General Practice,True,9.0,55
3,358-31-9711,04-02-2024 22:34,U,Titcombe,F,24,Native American/Alaska Native,General Practice,True,8.0,31
4,289-26-0537,04-09-2024 17:48,Y,Gionettitti,Male,5,African American,Orthopedics,False,NaN,10


In [6]:
# won't need patient name just id; verified proper name drop + also won't be using satisfaction score as a target
data = data.drop(['Patient First Inital', 'Patient Last Name', 'Patient Satisfaction Score'], axis=1)
data.head()

,Patient Id,Patient Admission Date,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Waittime
0,145-39-5406,20-03-2024 08:47,M,69,White,NaN,False,39
1,316-34-3057,15-06-2024 11:29,M,4,Native American/Alaska Native,NaN,True,27
2,897-46-3852,20-06-2024 09:13,F,56,African American,General Practice,True,55
3,358-31-9711,04-02-2024 22:34,F,24,Native American/Alaska Native,General Practice,True,31
4,289-26-0537,04-09-2024 17:48,Male,5,African American,Orthopedics,False,10


In [7]:
# see inconsistencies with how gender is indentified
print(data['Patient Gender'].unique())

['M' 'F' 'Male' 'Female']


In [8]:
# this fix works by finding the 'M' and 'F' in the gender column and replacing them with Male/Female
for x in data.index:
    if data.loc[x, 'Patient Gender'] == 'M':
        data.loc[x, 'Patient Gender'] = 'Male'
    if data.loc[x, 'Patient Gender'] == 'F':
        data.loc[x, 'Patient Gender'] = 'Female'
   
# verification      
print(data['Patient Gender'].unique())

['Male' 'Female']


In [ ]:
# need to change column names to fix the dates into days and times
data.rename({'Patient Id': 'patient_id', 
             'Patient Admission Date': 'admission_date',
             'Patient Gender': 'patient_gender',
             'Patient Age':  'patient_age',
             'Patient Race': 'patient_race',
             'Department Referral': 'department_referral',
             'Patient Admission Flag': 'admission_flag',
             'Patient Waittime': 'patient_waittime'}, axis=1, inplace=True)